In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "meta-llama/Llama-3.2-11B-Vision-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()
print("Model loaded successfully!")

In [ ]:
import json
from tqdm import tqdm

def load_medinst_dataset(input_file):
    """Load the MedInst dataset from JSON file (one JSON object per line)."""
    data = []
    with open(input_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

test_file = "datasets/MedInstQA/MedQa_test.json"
test_data = load_medinst_dataset(test_file)
print(f"Loaded {len(test_data)} test samples")

In [ ]:
def format_prompt(sample):
    """Format the MedInst sample into a prompt."""
    instruction = sample.get("instruction", "")
    input_text = sample.get("input", "")
    
    if instruction and input_text:
        prompt = f"{instruction}\n\n{input_text}"
    elif input_text:
        prompt = input_text
    else:
        prompt = instruction
    
    history = sample.get("history", [])
    if history:
        examples = "Here are some examples:\n\n"
        for i, (q, a) in enumerate(history, 1):
            examples += f"Example {i}:\n{q}\nAnswer: {a}\n\n"
        prompt = examples + "Now answer the following:\n" + prompt
    
    return prompt

def generate_response(prompt, model, tokenizer, max_new_tokens=256):
    """Generate response using Llama 3.2 model for text-only input."""
    
    messages = [
        {"role": "user", "content": prompt}
    ]
    
    input_text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )
    
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    input_length = inputs['input_ids'].shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return response.strip()

sample = test_data[0]
prompt = format_prompt(sample)
print("Sample prompt:")
print(prompt[:500] + "...")
print("\n" + "="*50)
response = generate_response(prompt, model, tokenizer)
print(f"\nModel response: {response}")
print(f"\nGround truth: {sample['output']}")

In [ ]:
def extract_answer(prediction, options_str):
    """Extract answer from prediction by matching against options."""
    prediction_lower = prediction.lower().strip()
    option_list = [opt.strip() for opt in options_str.split('/')]
    
    for option in option_list:
        if option.lower() in prediction_lower:
            return option
    return prediction

def run_inference(test_data, model, tokenizer, max_samples=None, save_every=100):
    """Run inference on test dataset."""
    results = []
    correct = 0
    total = 0
    
    if max_samples:
        test_data = test_data[:max_samples]
    
    for idx, sample in enumerate(tqdm(test_data, desc="Running inference")):
        instruction = sample.get("instruction", "")
        input_text = sample.get("input", "")
        ground_truth = sample.get("output", "")
        
        prompt = format_prompt(sample)
        
        try:
            prediction = generate_response(prompt, model, tokenizer)
            
            if "Options:" in input_text:
                options_str = input_text.split("Options:")[-1].strip()
                extracted_answer = extract_answer(prediction, options_str)
            else:
                extracted_answer = prediction
            
            is_correct = ground_truth.lower().strip() in prediction.lower()
            if is_correct:
                correct += 1
            total += 1
            
            result = {
                "id": idx,
                "instruction": instruction,
                "input": input_text,
                "ground_truth": ground_truth,
                "prediction": prediction,
                "extracted_answer": extracted_answer,
                "is_correct": is_correct
            }
            results.append(result)
            
            if (idx + 1) % 50 == 0:
                print(f"\nProgress: {idx + 1}/{len(test_data)} | Accuracy: {correct}/{total} ({100*correct/total:.2f}%)")
            
            if (idx + 1) % save_every == 0:
                with open("llama_3.2_medinst_predictions_partial.jsonl", "w") as f:
                    for r in results:
                        f.write(json.dumps(r) + "\n")
                        
        except Exception as e:
            print(f"Error processing sample {idx}: {e}")
            results.append({
                "id": idx,
                "error": str(e),
                "ground_truth": ground_truth
            })
    
    return results, correct, total

results, correct, total = run_inference(test_data, model, tokenizer, max_samples=None)

In [ ]:
output_file = "llama_3.2_vision_medinst_predictions.jsonl"
with open(output_file, "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

print(f"\n{'='*50}")
print(f"Final Results:")
print(f"{'='*50}")
print(f"Total samples: {total}")
print(f"Correct: {correct}")
print(f"Accuracy: {100*correct/total:.2f}%")
print(f"\nResults saved to: {output_file}")

summary = {
    "model": "Llama-3.2-11B-Vision-Instruct",
    "dataset": "MedInstQA/MedQa_test.json",
    "total_samples": total,
    "correct": correct,
    "accuracy": correct/total if total > 0 else 0
}

with open("llama_3.2_medinst_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"Summary saved to: llama_3.2_medinst_summary.json")